# ECE/CS 281b Final Project — Experiment Notebook

Loads the preprocessed tensors produced by `preprocessing.ipynb` and evaluates
four RobustBench models under human-vision-motivated degradations across five
severity levels. Computes Top-1 accuracy, Expected Calibration Error (ECE), and
Relative Accuracy Drop (RAD), then exports results to CSV and publication-ready figures.

ImageNet-C leaderboard results are cited directly from RobustBench and are
incorporated into the robustness ranking heatmap (Step 7) without runtime evaluation.

**Models evaluated (RobustBench ImageNet-C leaderboard):**

| Display name | RobustBench ID | Architecture | ImageNet-C Robust Acc |
|---|---|---|---|
| DeiT-B (Tian2022) | Tian2022Deeper_DeiT-B | ViT | 67.55% |
| DeiT-S (Tian2022) | Tian2022Deeper_DeiT-S | ViT | 62.91% |
| ResNet-50 (Hendrycks) | Hendrycks2020ManyFaces | CNN | 52.90% |
| ResNet-50 (AugMix) | Hendrycks2020AugMix | CNN | 49.33% |


In [ ]:
# RobustBench provides standardised model loading; timm supplies the ViT backbones.
!pip install robustbench timm -q

In [ ]:
!pip install git+https://github.com/fra31/auto-attack -q

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

---
## Step 0 — Imports & Configuration

Imports all dependencies, sets the global random seed for reproducibility,
configures I/O paths, and defines the ImageNet normalisation utility used
immediately before model inference.

In [ ]:
import random, warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from scipy.ndimage import gaussian_filter
from robustbench.utils import load_model as rb_load_model
from tqdm import tqdm

warnings.filterwarnings("ignore")

# ── Reproducibility ────────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

# ── Paths ──────────────────────────────────────────────────────────────────────
PREPROCESSED_DIR = Path("/content/drive/MyDrive/preprocessed")
HVM_D_DIR        = PREPROCESSED_DIR / "hvm_d"
# RESULTS_DIR      = Path("/content/results")
RESULTS_DIR      = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

# ── Experiment parameters ──────────────────────────────────────────────────────
SEVERITIES = list(range(1, 6))  # severity levels {1, 2, 3, 4, 5}
BATCH_SIZE = 128             

# ── ImageNet normalisation ─────────────────────────────────────────────────────
# Pre-computed as broadcast tensors for efficient batch application.
# Normalisation is applied just before inference; all saved .pt files are raw [0, 1].
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
_MEAN = torch.tensor(IMAGENET_MEAN).view(1, 3, 1, 1)
_STD  = torch.tensor(IMAGENET_STD).view(1, 3, 1, 1)

def normalize_batch(imgs: torch.Tensor) -> torch.Tensor:
    """Apply ImageNet channel-wise normalisation to a float32 [0, 1] batch.

    Args:
        imgs: Tensor of shape (N, 3, H, W), dtype float32, values in [0, 1].

    Returns:
        Normalised tensor of the same shape.
    """
    return (imgs - _MEAN) / _STD

print(f"Severities : {SEVERITIES}")
print(f"Batch size : {BATCH_SIZE}")


---
## Step 1 — Load Labels & Models

Loads ground-truth labels and instantiates the four RobustBench models.

`rb_load_model` returns a `NormalizedModel` wrapper that applies ImageNet
normalisation internally and expects raw \[0, 1\] inputs. To maintain a single,
explicit normalisation point consistent with how the preprocessed tensors are
stored, we unwrap the inner `nn.Module` via `.model` and apply `normalize_batch()`
manually in `run_inference()`. This avoids double-normalisation and keeps the
inference pipeline transparent.

In [ ]:
# PyTorch >= 2.0 sets weights_only=True by default in torch.load, which raises an
# UnpicklingError when loading legacy checkpoint files (e.g. RobustBench model weights
# and the preprocessed .pt tensors saved by preprocessing.ipynb).
import torch
_original_torch_load = torch.load
def _patched_torch_load(f, *args, **kwargs):
    kwargs['weights_only'] = False
    return _original_torch_load(f, *args, **kwargs)
torch.load = _patched_torch_load

In [ ]:
# ── Labels ────────────────────────────────────────────────────────────────────
# Shared across all degradation conditions; ordering matches the stratified subset
labels = torch.load("/content/drive/MyDrive/preprocessed/labels.pt")   # (5000,) int64
N = len(labels)
print(f"Labels loaded: {N} images")

# ── RobustBench Models ────────────────────────────────────────────────────────
# .model unwraps the NormalizedModel wrapper; the bare nn.Module expects
# ImageNet-normalised input, which is applied explicitly in run_inference().
ROBUSTBENCH_CONFIGS = {
    "DeiT-B (Tian2022)"       : ("Tian2022Deeper_DeiT-B", "ViT"),
    "DeiT-S (Tian2022)"       : ("Tian2022Deeper_DeiT-S", "ViT"),
    "ResNet-50 (Hendrycks)"  : ("Hendrycks2020ManyFaces",      "CNN"),
    "ResNet-50 (AugMix)"     : ("Hendrycks2020AugMix",    "CNN"),
}

models = {}
for display_name, (rb_name, arch_type) in ROBUSTBENCH_CONFIGS.items():
    print(f"Loading {display_name} ...", end=" ")
    rb_model = rb_load_model(rb_name, dataset='imagenet', threat_model='corruptions')
    model    = rb_model.model   # bare nn.Module, expects normalized input
    model.eval().to(DEVICE)
    n_params = sum(p.numel() for p in model.parameters()) / 1e6
    print(f"{n_params:.1f}M params  [{arch_type}]")
    models[display_name] = model

print(f"\n{len(models)} models ready.")


---
## Step 2 — Degradation Functions

Defines the five degradation functions used during preprocessing. They are
reproduced here to make this notebook self-contained.

In [ ]:
# ── Degradation functions ─────────────────────────────────────────────────────
# Each function accepts a single (3, H, W) float32 tensor in [0, 1] and returns
# a degraded tensor of the same shape and range. The rng argument accepts an
# integer seed for per-image reproducibility in stochastic functions.

def foveated_blur(img_t: torch.Tensor, severity: int, rng=None) -> torch.Tensor:
    img_np = img_t.numpy().transpose(1, 2, 0)
    H, W   = img_np.shape[:2]
    cx, cy = W / 2.0, H / 2.0
    y, x   = np.ogrid[:H, :W]
    dist   = np.sqrt((x - cx)**2 + (y - cy)**2)
    max_d  = np.sqrt(cx**2 + cy**2)
    sigma_map = (dist / max_d) * (severity * 2.0)
    n_levels  = 7
    sigmas    = np.linspace(0, severity * 2.0, n_levels)
    out       = np.zeros_like(img_np, dtype=np.float32)
    weights   = np.zeros((H, W), dtype=np.float32)
    bw        = max(severity * 2.0 / n_levels, 0.5)  # blending bandwidth
    for s in sigmas:
        blurred  = gaussian_filter(img_np.astype(np.float32), sigma=[s, s, 0])
        w        = np.exp(-0.5 * ((sigma_map - s) / bw)**2).astype(np.float32)
        out     += blurred * w[:, :, None]
        weights += w
    out = np.clip(out / (weights[:, :, None] + 1e-8), 0, 1)  # normalise weights
    return torch.from_numpy(out.transpose(2, 0, 1).copy())

def contrast_loss(img_t: torch.Tensor, severity: int, rng=None) -> torch.Tensor:
    factors = [0.93, 0.80, 0.65, 0.50, 0.38]
    return torch.clamp(0.5 + (img_t - 0.5) * factors[severity - 1], 0, 1)

def shot_noise(img_t: torch.Tensor, severity: int, rng=None) -> torch.Tensor:
    scales = [5000, 1000, 300, 100, 50]
    _rng   = np.random.default_rng(rng)
    img_np = img_t.numpy()
    noisy  = _rng.poisson(img_np * scales[severity - 1]).astype(np.float32) / scales[severity - 1]
    return torch.from_numpy(np.clip(noisy, 0, 1))

def fixation_jitter(img_t: torch.Tensor, severity: int, rng=None) -> torch.Tensor:
    _rng   = np.random.default_rng(rng)
    dx     = int(_rng.integers(-severity, severity + 1))
    dy     = int(_rng.integers(-severity, severity + 1))
    img_np = img_t.numpy().transpose(1, 2, 0)
    H, W   = img_np.shape[:2]
    pad_h, pad_w = abs(dy), abs(dx)
    img_pad = np.pad(img_np, ((pad_h, pad_h), (pad_w, pad_w), (0, 0)), mode='edge')
    y0, x0  = pad_h - dy, pad_w - dx  # dy > 0 → shift down → crop origin moves up
    return torch.from_numpy(img_pad[y0:y0+H, x0:x0+W, :].astype(np.float32).transpose(2, 0, 1).copy())

def composed_pipeline(img_t: torch.Tensor, severity: int, rng=None) -> torch.Tensor:
    _rng = np.random.default_rng(rng)
    img  = foveated_blur(img_t,  severity, _rng.integers(0, 2**31))
    img  = contrast_loss(img,    severity)
    img  = shot_noise(img,       severity, _rng.integers(0, 2**31))
    img  = fixation_jitter(img,  severity, _rng.integers(0, 2**31))
    return img

DEGRADATIONS = {
    "foveated_blur"  : foveated_blur,
    "contrast_loss"  : contrast_loss,
    "shot_noise"     : shot_noise,
    "fixation_jitter": fixation_jitter,
    "composed"       : composed_pipeline,
}
print("Degradation functions defined:", list(DEGRADATIONS.keys()))


---
## Step 3 — Inference Utility & Clean Baseline

Defines `run_inference()`, which loads a preprocessed float16 tensor, applies
ImageNet normalisation, and runs batched forward passes under `torch.no_grad()`.
Returns per-sample maximum softmax confidence, predicted class index, and ground-truth
label as NumPy arrays for metric computation.

The clean baseline is evaluated first to establish the per-model `acc_clean`
reference used in the RAD computation (Step 6).

In [ ]:
@torch.no_grad()
def run_inference(model, tensor_path, labels, device, batch_size=128):
    """Load a preprocessed tensor and run batched inference.

    Loads a raw [0, 1] float16 tensor from disk, casts to float32, applies
    ImageNet normalisation, and performs forward passes in mini-batches.

    Args:
        model:       An nn.Module in eval mode expecting normalised input.
        tensor_path: Path to a .pt file of shape (N, 3, 224, 224), float16, [0, 1].
        labels:      Ground-truth class indices, shape (N,).
        device:      Target torch.device for inference.
        batch_size:  Number of images per forward pass (default: 128).

    Returns:
        Tuple of (confidences, predictions, labels) as float32/int64 NumPy arrays,
        each of shape (N,). Confidences are maximum softmax probabilities.
    """
    imgs = torch.load(tensor_path).float()   # float16 → float32, values remain in [0, 1]
    imgs = normalize_batch(imgs)             # apply ImageNet channel-wise normalisation

    all_confs, all_preds = [], []
    for start in range(0, len(imgs), batch_size):
        batch  = imgs[start:start+batch_size].to(device)
        logits = model(batch)
        probs  = F.softmax(logits, dim=-1)
        confs, preds = probs.max(dim=-1)     # top-1 confidence and predicted class
        all_confs.append(confs.cpu())
        all_preds.append(preds.cpu())

    return (
        torch.cat(all_confs).numpy(),
        torch.cat(all_preds).numpy(),
        labels.numpy(),
    )

print("run_inference defined.")

# ── Clean baseline ────────────────────────────────────────────────────────────
# Evaluates each model on unmodified images to establish acc_clean,
clean_results = {}
print("Running clean baseline...")
for model_name, model in models.items():
    c, p, l = run_inference(model, PREPROCESSED_DIR / "clean.pt", labels, DEVICE, BATCH_SIZE)
    clean_results[model_name] = {"confs": c, "preds": p, "labels": l}
    acc = (p == l).mean()
    print(f"  {model_name:<30} clean accuracy = {acc:.4f}")


---
## Step 4 — Human-Vision-Motivated Degradations: Severity Sweep

Iterates over all 4 models × 5 degradations × 5 severity levels = 100 inference
runs. Results are accumulated in `sweep_results`, a nested dict keyed by
`[model_name][degradation_name][severity]`, each value containing confidence,
prediction, and label arrays.

In [ ]:
sweep_results = defaultdict(lambda: defaultdict(dict))

total_runs = len(models) * len(DEGRADATIONS) * len(SEVERITIES)
print(f"Total inference runs: {total_runs}\n")

for model_name, model in models.items():
    print(f"── {model_name} ──")
    for deg_name in tqdm(DEGRADATIONS, desc="  degradations", leave=False):
        for sev in SEVERITIES:
            path = HVM_D_DIR / f"{deg_name}_sev{sev}.pt"
            c, p, l = run_inference(model, path, labels, DEVICE, BATCH_SIZE)
            sweep_results[model_name][deg_name][sev] = {"confs": c, "preds": p, "labels": l}

print("\nSeverity sweep complete.")


---
## Step 5 — ImageNet-C Results (RobustBench Leaderboard)

ImageNet-C evaluation is skipped at runtime. Results are cited directly from the  
[RobustBench ImageNet-C Leaderboard](https://robustbench.github.io/).

| Model | Architecture | ImageNet-C Robust Acc |
|---|---|---|
| Tian2022 DeiT-B | ViT | 67.55% |
| Tian2022 DeiT-S | ViT | 62.91% |
| Hendrycks2020ManyFaces ResNet-50 | CNN | 52.90% |
| Hendrycks2020AugMix ResNet-50 | CNN | 49.33% |

These values are used in the robustness ranking heatmap below.


---
## Step 6 — Compute Metrics

Computes three metrics per (model, degradation, severity) triple and assembles
them into a tidy DataFrame exported to `all_results.csv`.

- **Accuracy** — Top-1 classification accuracy: fraction of samples where the
  argmax prediction matches the ground-truth label.
- **ECE** — Expected Calibration Error with 15 equal-width confidence bins
  (Guo et al., 2017). Lower values indicate better alignment between predicted
  confidence and empirical accuracy.
- **RAD** — Relative Accuracy Drop: `(acc_clean − acc_corrupted) / acc_clean`.
  Normalises degradation sensitivity to each model's own clean baseline,
  enabling fair cross-architecture comparison independent of absolute accuracy.

In [ ]:
def compute_accuracy(preds: np.ndarray, labels: np.ndarray) -> float:
    """Return Top-1 accuracy as a scalar in [0, 1]."""
    return float((preds == labels).mean())

def compute_ece(confidences: np.ndarray, preds: np.ndarray,
                labels: np.ndarray, n_bins: int = 15) -> float:
    """Compute Expected Calibration Error with equal-width confidence bins.

    Partitions samples into n_bins bins over [0, 1] by maximum softmax confidence,
    then computes the confidence-weighted mean absolute gap between bin accuracy
    and bin mean confidence (Guo et al., 2017).

    Args:
        confidences: Maximum softmax probability per sample, shape (N,).
        preds:       Predicted class indices, shape (N,).
        labels:      Ground-truth class indices, shape (N,).
        n_bins:      Number of equal-width bins (default: 15).

    Returns:
        ECE as a scalar in [0, 1]. Lower values indicate better calibration.
    """
    correct = (preds == labels).astype(float)
    bins    = np.linspace(0, 1, n_bins + 1)
    ece     = 0.0
    for lo, hi in zip(bins[:-1], bins[1:]):
        mask = (confidences > lo) & (confidences <= hi)
        if mask.sum() == 0:
            continue
        ece += mask.mean() * abs(correct[mask].mean() - confidences[mask].mean())
    return float(ece)

def compute_rad(acc_corrupted: float, acc_clean: float) -> float:
    """Compute Relative Accuracy Drop (RAD).

    A value of 0 indicates no degradation; 1 indicates complete accuracy collapse. 
    Returns NaN if acc_clean is zero.
    """
    if acc_clean == 0:
        return float('nan')
    return float((acc_clean - acc_corrupted) / acc_clean)

# ── Assemble results DataFrame ────────────────────────────────────────────────
# Clean baseline rows use severity=0 and RAD=0 as sentinel values.
rows = []

for model_name in models:
    d         = clean_results[model_name]
    acc_clean = compute_accuracy(d["preds"], d["labels"])
    rows.append({
        "model": model_name, 
        "degradation": "clean", 
        "severity": 0,
        "accuracy"  : acc_clean,
        "ece"       : compute_ece(d["confs"], d["preds"], d["labels"]),
        "rad"       : 0.0,
    })
    for deg_name in DEGRADATIONS:
        for sev in SEVERITIES:
            d    = sweep_results[model_name][deg_name][sev]
            acc  = compute_accuracy(d["preds"], d["labels"])
            ece  = compute_ece(d["confs"], d["preds"], d["labels"])
            rad  = compute_rad(acc, acc_clean)
            rows.append({
                "model": model_name, "degradation": deg_name, "severity": sev,
                "accuracy": acc, "ece": ece, "rad": rad,
            })

results_df = pd.DataFrame(rows)
results_df.to_csv(RESULTS_DIR / "all_results.csv", index=False)

conf_records = []
for model_name in models:
    # clean baseline
    d = clean_results[model_name]
    for i in range(len(d["confs"])):
        conf_records.append({
            "model"      : model_name,
            "degradation": "clean",
            "severity"   : 0,
            "confidence" : float(d["confs"][i]),
            "correct"    : int(d["preds"][i] == d["labels"][i]),
        })
    # sweep
    for deg_name in DEGRADATIONS:
        for sev in SEVERITIES:
            d = sweep_results[model_name][deg_name][sev]
            for i in range(len(d["confs"])):
                conf_records.append({
                    "model"      : model_name,
                    "degradation": deg_name,
                    "severity"   : sev,
                    "confidence" : float(d["confs"][i]),
                    "correct"    : int(d["preds"][i] == d["labels"][i]),
                })

conf_df = pd.DataFrame(conf_records)
conf_df.to_csv(RESULTS_DIR / "raw_confidences.csv", index=False)
print(f"Saved raw_confidences.csv — {len(conf_df):,} rows")

print(f"Results shape : {results_df.shape}")
print(f"Saved → {RESULTS_DIR / 'all_results.csv'}")

---
## Step 7 — Visualisation

Generates four figures, each saved to `RESULTS_DIR`:

| Figure | File | Description |
|---|---|---|
| Plot 1 | `accuracy_vs_severity.png` | Top-1 accuracy vs. severity for each degradation |
| Plot 2 | `ece_vs_severity.png` | ECE vs. severity for each degradation |
| Plot 3 | `rad_vs_severity.png` | RAD vs. severity for each degradation |
| Plot 4 | `robustness_heatmap.png` | Mean accuracy heatmap with ImageNet-C leaderboard column |
| Plot 5 | `Reliability Diagram.png` | Gap bars indicate areas of miscalibration |

In [ ]:
MODEL_COLORS = {
    "DeiT-B (Tian2022)"      : "#4e79a7",
    "DeiT-S (Tian2022)"      : "#76b7b2",
    "ResNet-50 (Hendrycks)" : "#e15759",
    "ResNet-50 (AugMix)"    : "#f28e2b",
}

deg_names = list(DEGRADATIONS.keys())

# ── Plot 1: Accuracy vs Severity ──────────────────────────────────────────────
fig, axes = plt.subplots(1, len(deg_names), figsize=(20, 4), sharey=True)
for ax, deg_name in zip(axes, deg_names):
    for model_name in models:
        sub      = results_df[(results_df.model == model_name) &
                               (results_df.degradation == deg_name)].sort_values("severity")
        baseline = results_df[(results_df.model == model_name) &
                               (results_df.degradation == "clean")]
        ax.plot([0] + list(sub.severity),
                [baseline.accuracy.values[0]] + list(sub.accuracy),
                label=model_name, color=MODEL_COLORS[model_name],
                marker="o", linewidth=2, markersize=5)
    ax.set_title(deg_name.replace("_", " ").title(), fontsize=10)
    ax.set_xlabel("Severity")
    ax.set_xticks(range(0, 6))
    ax.set_ylim(0, 1)
    ax.grid(True, alpha=0.3)
axes[0].set_ylabel("Top-1 Accuracy")
axes[-1].legend(loc="lower left", fontsize=8)
plt.suptitle("Accuracy vs. Severity — Human-Vision-Motivated Degradations", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "accuracy_vs_severity.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# ── Plot 2: ECE vs Severity ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, len(deg_names), figsize=(20, 4), sharey=True)
for ax, deg_name in zip(axes, deg_names):
    for model_name in models:
        sub      = results_df[(results_df.model == model_name) &
                               (results_df.degradation == deg_name)].sort_values("severity")
        baseline = results_df[(results_df.model == model_name) &
                               (results_df.degradation == "clean")]
        ax.plot([0] + list(sub.severity),
                [baseline.ece.values[0]] + list(sub.ece),
                label=model_name, color=MODEL_COLORS[model_name],
                marker="s", linewidth=2, markersize=5)
    ax.set_title(deg_name.replace("_", " ").title(), fontsize=10)
    ax.set_xlabel("Severity")
    ax.set_xticks(range(0, 6))
    ax.grid(True, alpha=0.3)
axes[0].set_ylabel("ECE (↓ better)")
axes[-1].legend(loc="upper left", fontsize=8)
plt.suptitle("ECE vs. Severity — Human-Vision-Motivated Degradations", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "ece_vs_severity.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# ── Plot 3: RAD vs Severity ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, len(deg_names), figsize=(20, 4), sharey=True)
for ax, deg_name in zip(axes, deg_names):
    for model_name in models:
        sub = results_df[(results_df.model == model_name) &
                          (results_df.degradation == deg_name)].sort_values("severity")
        ax.plot(sub.severity, sub.rad,
                label=model_name, color=MODEL_COLORS[model_name],
                marker="D", linewidth=2, markersize=5)
    ax.set_title(deg_name.replace("_", " ").title(), fontsize=10)
    ax.set_xlabel("Severity")
    ax.set_xticks(SEVERITIES)
    ax.set_ylim(0, 1)
    ax.grid(True, alpha=0.3)
axes[0].set_ylabel("RAD — Relative Accuracy Drop (↑ worse)")
axes[-1].legend(loc="upper left", fontsize=8)
plt.suptitle("Relative Accuracy Drop (RAD) vs. Severity — Human-Vision-Motivated Degradations", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "rad_vs_severity.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── Plot 4: Robustness Ranking Heatmap ───────────────────────────────────────
# Aggregates mean Top-1 accuracy over severity levels 1–5 per (model, degradation),
# appends the ImageNet-C leaderboard column, and ranks models by mean HVM accuracy.
# ImageNet-C values are cited from the RobustBench leaderboard (no runtime evaluation).
IMAGENET_C_LEADERBOARD = {
    "DeiT-B (Tian2022)"      : 0.6755,
    "DeiT-S (Tian2022)"      : 0.6291,
    "ResNet-50 (Hendrycks)" : 0.5290,
    "ResNet-50 (AugMix)"    : 0.4933,
}

ranking_df = (
    results_df[results_df.degradation != "clean"]
    .groupby(["model", "degradation"])["accuracy"]
    .mean().unstack("degradation").round(4)
)
ranking_df["imagenet_c (leaderboard)"] = pd.Series(IMAGENET_C_LEADERBOARD).round(4)

hvm_cols  = list(DEGRADATIONS.keys())
all_cols  = hvm_cols + ["imagenet_c (leaderboard)"]
ranking_df["mean_hvm"] = ranking_df[hvm_cols].mean(axis=1)
ranking_df = ranking_df.sort_values("mean_hvm", ascending=False)

print("\n── Robustness Ranking (mean Top-1 accuracy, severity 1–5) ──")
print(ranking_df[all_cols + ["mean_hvm"]].to_string())
ranking_df.to_csv(RESULTS_DIR / "robustness_ranking.csv")

plot_df = ranking_df[all_cols + ["mean_hvm"]]

fig, ax = plt.subplots(figsize=(14, 3))
im = ax.imshow(plot_df.values, cmap="RdYlGn", aspect="auto", vmin=0, vmax=1)
ax.set_xticks(range(len(plot_df.columns)))
ax.set_xticklabels([c.replace("_", "\n") for c in plot_df.columns], fontsize=9)
ax.set_yticks(range(len(plot_df.index)))
ax.set_yticklabels(plot_df.index, fontsize=10)
for i in range(len(plot_df.index)):
    for j in range(len(plot_df.columns)):
        val   = plot_df.values[i, j]
        color = "black" if val > 0.45 else "white"  # ensure legibility on dark cells
        ax.text(j, i, f"{val:.3f}", ha="center", va="center",
                fontsize=9, color=color, fontweight="bold")
# Vertical divider separating evaluated HVM columns from leaderboard/mean columns
ax.axvline(x=len(hvm_cols) - 0.5, color="black", linewidth=2.5)
ax.annotate("← human-vision-motivated degradations (evaluated) →",
            xy=((len(hvm_cols)-1)/2 / len(all_cols+['mean_hvm']), 1.18),
            xycoords="axes fraction", ha="center", fontsize=9, color="#555555",
            annotation_clip=False)
ax.annotate("Leaderboard / Mean",
            xy=((len(hvm_cols)+0.8) / len(all_cols+['mean_hvm']), 1.18),
            xycoords="axes fraction", ha="center", fontsize=9, color="#555555",
            annotation_clip=False)
plt.colorbar(im, ax=ax, label="Top-1 Accuracy", pad=0.01)
ax.set_title("Robustness Ranking Heatmap", fontsize=11, pad=28)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "robustness_heatmap.png", dpi=150)
plt.show()
print("Saved → results/robustness_heatmap.png")


In [ ]:
# ── Plot 5: Reliability Diagram ───────────────────────────────────────────────────────
conf_df = pd.read_csv(RESULTS_DIR / "raw_confidences.csv")

def plot_reliability(ax, confidences, correct, label, color, n_bins=15):
    """
    Draw a reliability curve and return ECE.
    Gap bars indicate areas of miscalibration (overconfidence = bars below the diagonal).
    """
    bins = np.linspace(0, 1, n_bins + 1)
    bin_accs, bin_confs, bin_counts = [], [], []

    for lo, hi in zip(bins[:-1], bins[1:]):
        mask = (confidences > lo) & (confidences <= hi)
        if mask.sum() == 0:
            bin_accs.append(np.nan)
            bin_confs.append((lo + hi) / 2)
            bin_counts.append(0)
        else:
            bin_accs.append(correct[mask].mean())
            bin_confs.append(confidences[mask].mean())
            bin_counts.append(mask.sum())

    bin_accs   = np.array(bin_accs)
    bin_confs  = np.array(bin_confs)
    bin_counts = np.array(bin_counts)

    # Gap bars (Red = Overconfidence, Blue = Overly Conservative)
    for i, (conf, acc, cnt) in enumerate(zip(bin_confs, bin_accs, bin_counts)):
        if np.isnan(acc) or cnt == 0:
            continue
        gap_color = "#e15759" if acc < bins[i] else "#4e79a7"
        ax.bar(bins[i], abs(acc - bins[i]),
               width=bins[i+1] - bins[i],
               bottom=min(acc, bins[i]),
               alpha=0.25, color=gap_color, align='edge')

    # Reliability curve
    valid = ~np.isnan(bin_accs)
    ax.plot(bin_confs[valid], bin_accs[valid],
            marker='o', color=color, linewidth=2,
            markersize=5, label=label)

    # ECE
    total = bin_counts[bin_counts > 0].sum()
    ece = float(np.nansum(
        (bin_counts / total) * np.abs(bin_accs - bin_confs) * (bin_counts > 0)
    ))
    return ece


fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ── DeiT-B，severity 0→4 ────────────────────────────────────────
ax = axes[0]
ax.plot([0,1], [0,1], 'k--', lw=1, alpha=0.4, label='Perfect calibration')

conditions = [
    ("clean",         0, "#aec7e8", "Clean (sev=0)"),
    ("contrast_loss", 1, "#7fb5d5", "Contrast Loss sev=1"),
    ("contrast_loss", 2, "#4e79a7", "Contrast Loss sev=2"),
    ("contrast_loss", 3, "#c44e52", "Contrast Loss sev=3"),
    ("contrast_loss", 4, "#8b0000", "Contrast Loss sev=4"),
]
for deg, sev, color, label in conditions:
    sub = conf_df[
        (conf_df.model == "DeiT-B (Tian2022)") &
        (conf_df.degradation == deg) &
        (conf_df.severity == sev)
    ]
    ece = plot_reliability(ax, sub.confidence.values,
                           sub.correct.values, label, color)
    ax.lines[-1].set_label(f"{label}  (ECE={ece:.3f})")

ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.set_xlabel("Confidence", fontsize=10)
ax.set_ylabel("Accuracy", fontsize=10)
ax.set_title("DeiT-B — Calibration Drift\nacross Contrast Loss Severities",
             fontsize=10)
ax.legend(fontsize=8, loc="upper left")
ax.grid(True, alpha=0.3)

# ── Contrast Loss sev=4，Comparison of four models ───────────────────────────────────
ax = axes[1]
ax.plot([0,1], [0,1], 'k--', lw=1, alpha=0.4, label='Perfect calibration')

conditions = [
    ("DeiT-B (Tian2022)",     "#4e79a7", "DeiT-B"),
    ("DeiT-S (Tian2022)",     "#76b7b2", "DeiT-S"),
    ("ResNet-50 (Hendrycks)", "#e15759", "ResNet-50 (Hendrycks)"),
    ("ResNet-50 (AugMix)",    "#f28e2b", "ResNet-50 (AugMix)"),
]
for model_name, color, label in conditions:
    sub = conf_df[
        (conf_df.model == model_name) &
        (conf_df.degradation == "contrast_loss") &
        (conf_df.severity == 4)
    ]
    ece = plot_reliability(ax, sub.confidence.values,
                           sub.correct.values, label, color)
    ax.lines[-1].set_label(f"{label}  (ECE={ece:.3f})")

ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.set_xlabel("Confidence", fontsize=10)
ax.set_ylabel("Accuracy", fontsize=10)
ax.set_title("Contrast Loss sev=4\nAll Models Compared", fontsize=10)
ax.legend(fontsize=8, loc="upper left")
ax.grid(True, alpha=0.3)

plt.suptitle("Reliability Diagrams — Confidence Calibration under Contrast Loss",
             fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "reliability_diagram.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → results/reliability_diagram.png")